# Qwen3-8B on Colab: first test

**Goal of this notebook:** prove that a small model running on a Colab GPU can read a piece of a lecture
transcript and answer with clean JSON facts. That is the real job of our "digest" stage.

No API, no server here. Just text in, text out.

**The steps**

1. Look at the GPU we got
2. Connect Google Drive (our permanent storage)
3. Download the model into Drive once (about 5 GB)
4. Copy it to Colab's fast local disk
5. Install the engine that runs the model (`llama-cpp-python`)
6. Load the model into the GPU
7. Say hello to it
8. Give it a real transcript slice with our real prompt
9. Check that the answer is valid JSON
10. Save the result to Drive so we can compare later

Run the cells from top to bottom. Read the text above each cell before you run it.

## Step 1: Which GPU did we get?

The model is big, so it must run on a GPU. A CPU would be about 20 times slower.

`nvidia-smi` is a tool that is already installed on Colab. It prints the GPU name and how much memory it has.
The `!` at the start of the line tells Colab: "run this as a terminal command, not as Python".

If you get an error here, you have no GPU: use the menu **Runtime > Change runtime type** and pick a GPU
(T4 or L4), then run the cell again.

In [ ]:
!nvidia-smi

## Step 2: Connect Google Drive

Colab's disk is wiped every time the session ends. Google Drive is not.
So anything we want to keep (the model file, our test results) goes into Drive.

When you run this cell a pop-up asks for permission. Click **Allow**.

In [ ]:
from google.colab import drive    # Colab's own helper for Drive

drive.mount("/content/drive")     # after this, your Drive looks like a normal folder at /content/drive/MyDrive

Now we make our own folder inside Drive, so we do not mix our files with anything else.

In [ ]:
import os    # "os" lets Python create folders and look at files

# one main folder, and two folders inside it
DRIVE_FOLDER = "/content/drive/MyDrive/AQ-Worker"
MODEL_FOLDER = DRIVE_FOLDER + "/models"    # the big model file will live here
RUN_FOLDER = DRIVE_FOLDER + "/runs"        # our test results will be saved here

# exist_ok=True means: if the folder is already there, that is fine, do not complain
os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(RUN_FOLDER, exist_ok=True)

print("Drive folder is ready:", DRIVE_FOLDER)

## Step 3: Download the model into Drive (only once)

**Which model?** `Qwen3-8B`. It has 8 billion parameters, which is small enough to be fast and to fit easily
in 16 GB of GPU memory, but still good at following "answer in JSON" instructions.

**What is `Q4_K_M`?** The original model uses 16 bits for every number, which is about 16 GB.
Quantization squeezes each number into about 4 bits. The file shrinks to roughly 5 GB, runs faster,
and loses only a little quality. `Q4_K_M` is the usual balanced choice.

**What is `.gguf`?** The file format that `llama.cpp` (our engine) reads.

The download goes **straight into Drive**, so next session we skip it.

In [ ]:
%pip install -q huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download    # downloads one file from huggingface.co

REPO_NAME = "Qwen/Qwen3-8B-GGUF"       # the page on huggingface that holds the file
FILE_NAME = "Qwen3-8B-Q4_K_M.gguf"     # the exact file we want from that page

path_on_drive = MODEL_FOLDER + "/" + FILE_NAME

if os.path.exists(path_on_drive):
    print("The model is already in Drive. Nothing to download.")
else:
    print("Downloading about 5 GB. This takes a few minutes...")
    hf_hub_download(repo_id=REPO_NAME, filename=FILE_NAME, local_dir=MODEL_FOLDER)
    print("Download finished.")

# a quick sanity check: the file should be around 5 GB, a tiny file means the download broke
size_gb = os.path.getsize(path_on_drive) / 1e9    # bytes -> gigabytes
print("File:", path_on_drive)
print(f"Size: {size_gb:.1f} GB")

**Only run the next cell if the download cell failed with "not found".**
It prints the real `.gguf` file names in that repository, so you can fix `FILE_NAME` above.

In [ ]:
from huggingface_hub import list_repo_files

for name in list_repo_files(REPO_NAME):
    if name.endswith(".gguf"):
        print(name)

## Step 4: Copy the model to Colab's local disk

Drive is a network drive. Reading 5 GB from it every time is slow.
Colab's own disk (`/content`) is much faster, so we copy the file there once per session.
It takes about a minute. The copy disappears when the session ends, but the original stays in Drive.

In [ ]:
import shutil    # "shutil" copies files

os.makedirs("/content/models", exist_ok=True)
path_on_colab = "/content/models/" + FILE_NAME

if os.path.exists(path_on_colab):
    print("Already copied in this session.")
else:
    print("Copying from Drive to the local disk...")
    shutil.copyfile(path_on_drive, path_on_colab)
    print("Copy finished.")

## Step 5: Install the engine

A `.gguf` file is just numbers. We need a program that can run them: `llama.cpp`.
`llama-cpp-python` is that program wrapped so we can call it from Python.

We install a version that is **already compiled for GPUs** (the `cu124` link), so it takes seconds, not minutes.

In [ ]:
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import llama_cpp

print("version:", llama_cpp.__version__)
print("can use the GPU:", llama_cpp.llama_supports_gpu_offload())    # we need this to say True

**If it says `False` or the install failed:** run the next cell. It builds the engine from source with GPU
support switched on. That takes about 10 to 15 minutes, so only do it if you have to.
After it finishes, use **Runtime > Restart session**, then run the cells again from Step 2.

In [ ]:
# ONLY if the cell above said False or failed. Takes 10 to 15 minutes.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir --force-reinstall

## Step 6: Load the model into the GPU

The settings:

- `model_path`: the file we copied in Step 4.
- `n_gpu_layers=-1`: put **all** layers of the model on the GPU. (A number like 20 would put only 20 there.)
- `n_ctx=8192`: the context window. It is how much text the model can hold in its head at once,
  counted in tokens (a token is about 4 characters). Our prompt is about 1,500 tokens and the answer can be up to 1,500 more, so 8192 is plenty.
- `verbose=False`: do not print the engine's noisy logs.

Loading takes about 10 to 30 seconds.

In [ ]:
import time
from llama_cpp import Llama

start = time.time()

llm = Llama(
    model_path=path_on_colab,
    n_gpu_layers=-1,
    n_ctx=8192,
    verbose=False,
)

print(f"Model loaded in {time.time() - start:.0f} seconds")

## Step 7: A helper function, and a hello test

We will talk to the model many times, so we write the talking part once, as a function called `ask`.

A chat model receives a list of messages:

- the **system** message sets the rules ("you are a note-taking assistant, answer in JSON")
- the **user** message holds the actual request

What the function does, in order:

1. start a stopwatch
2. send both messages to the model
3. read the text that came back, and how many tokens the model wrote
4. clean the text
5. give back three things: the text, the token count and the seconds it took

Two details:

- `temperature=0.2` means "be steady, do not get creative". A low value gives more repeatable JSON.
- Qwen3 can "think out loud" in `<think>...</think>` tags before it answers. We add `/no_think` to our message to
  switch that off (thinking is slow and we do not need it), and we also delete any leftover think block from the answer.

In [ ]:
import re

def ask(system_text, user_text, max_tokens=1500):
    start = time.time()

    reply = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text + "\n/no_think"},
        ],
        max_tokens=max_tokens,    # stop after this many tokens, so it can never run forever
        temperature=0.2,
    )

    seconds = time.time() - start
    text = reply["choices"][0]["message"]["content"]
    tokens = reply["usage"]["completion_tokens"]

    # delete a <think>...</think> block if one is there (re.S lets "." also match line breaks)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

    return text, tokens, seconds

In [ ]:
text, tokens, seconds = ask("You are a friendly assistant.", "Say hello in one short sentence.")

print(text)
print(f"({tokens} tokens in {seconds:.1f} seconds)")

If you see a greeting, **the model works on your Colab GPU**. The first call is a little slower than the next ones.

## Step 8: The real task, with our real prompt

Now the same input our app gives the cloud model. Three pieces. They are filled in already, you only need to read them.

1. **System prompt:** the rules for the model.
2. **User prompt template:** the task, the JSON shape we want, and a placeholder `{{CHUNK}}`.
3. **Transcript slice:** about 5,000 characters of a real lecture, slice 3 of 44, starting at 09:38.
   (Our app cuts a long transcript into slices like this and runs one digest per slice.)

In [ ]:
SYSTEM_PROMPT = r'''You are the first stage of a lecture-note pipeline. You read ONE slice of a transcript and extract ONLY its teachable knowledge as compact, faithful facts that a second stage will turn into notebook pages.

Drop intros, sponsors, subscribe asks, filler, jokes, tangents and anecdotes with no teaching value. Do not invent or add outside facts; fix only obvious auto-caption spelling of technical terms. Write facts in your own short words (no long verbatim quotes), in English. Keep numbers, names, commands, versions and code exactly as stated. Use the [mm:ss] markers for `start`.
Output ONE JSON object only.'''

In [ ]:
USER_TEMPLATE = r'''Lecture: title: System Design Crash Course in 4 Hours; channel: Mohit Chhabra; length: 3:45:12
This is slice 3 of 44. (Slices overlap slightly at the edges: do not worry about a cut-off first/last sentence.)

Extract the teachable knowledge as 1-6 topics, in order. Total output must stay under about 181 words, so be selective and dense: prefer facts, definitions, steps, numbers and code over narration. If the slice contains NO teachable content (pure intro/outro/sponsor), return {"topics":[]}.

JSON shape (omit empty/null optional fields):
{"topics":[{"heading":"short topic name","start":"mm:ss","facts":["one self-contained fact per string"],"terms":[{"term":"x","definition":"y"}],"steps":["ordered step, only if a process was explained"],"example":"one illustrative example or null","code":[{"language":"py","code":"only code actually shown"}],"table":{"headers":["a","b"],"rows":[["x","y"]]} or null,"formula":"only if stated or null","diagram_hint":"if a process/architecture/comparison exists: type (flow|stack|compare|cycle|tree|timeline|mindmap|sequence) + its parts, else null"}]}

TRANSCRIPT SLICE:
"""
{{CHUNK}}
"""'''

In [ ]:
TRANSCRIPT_SLICE = r'''[09:38] Like create, read, update, delete. So these major operations are performed on the database because it contains the data. You can create more data. You can read the data. Whatever existing data is present, you can read that. Then you can update some of the data or you can also delete some of the data. Okay. So this is a very basic overview like how things work, right? And maybe most of you already know this flow. Okay. So this part basically request and response. [10:11] This is basically API, right? So I can write it here. Uh you already may know this. There is client. There is server uh there is database then there is API so in API when client is asking something from the server it's called request and when server gives back the output it's called the response okay very basic overview right so let's move forward with it because we want to get more details in it so our major flow in our question is related to this side you can say uh like we simply said from [10:46] client a request will go to the server and server will give back the response, right? But what actually happens in between like how does this all happen? Uh so before going to that part, uh generally like if you want to know the request and response in what format it is, so most of the times it is in JSON uh format. So JSON is used in most of the APIs you would see that's very famous. There are other options as well to send like the request response format but JSON is mostly used. So just to [11:19] showcase you the request response it would be something like this. So here you can see it's a get uh API method and then this is basically the URL and we are using http 1.1 here. Don't worry about all this. I will cover all this information in detail in this video. Then this is the host uh api.example.com. You can imagine it something like www.google.com or youtube.com Facebook.com everything all such things are host. So here the host is api.acample.com. [11:49] Uh then there is authorization content type accept user agent. User agent will basically mean uh from where you are hitting this API. So for example I ran this from postman. So it says postman and here you can see it says JSON. So JSON as I mentioned you it's like a very famous um way in which you send the request in the response. Okay. Um and rest of the stuff don't worry uh we will discuss about each of these things. Um but basically is like host and URL will [12:21] give you like where are you trying to hit the API. HTTP is a protocol. Get is a method. There are other methods as well like post, put, patch, delete uh which can be used for API. And then authorization it's basically to authorize the end user like who is making the API call to authenticate that user we use the authorization which is part of the headers. Um we'll talk about it later don't worry. So this was an example of what client will send to the server that is the request. Now let me [12:52] also show you an example of the response that is what server will send back to the client. Okay. So this is the example of the response. Okay. So first thing is it tells about the protocol same as like HTTP 1.1. Uh then 200 is the status code which basically tells you it was a successful API call that whatever request was sent it was successfully returned as a response. So whatever request was made successful response has been sent from the server side. Okay. [13:24] Again there is content type which is JSON. Uh cache control is there which talks about no cache or cache. You can ignore it for now. Then basically you can see this is the response that is received. It provides a lot of information in it and basically this format is in JSON. Okay. So this is the like just an example of how a request and response looks like. Okay. So let me remove all this and then we can go further in detail about all the components that will be required in the API. So what we are going to talk about is first is the network layer. Okay. So [14:00] network layer will contain the components like pers DNS. Then there is TCP protocol. Then there is TLS protocol that you should be aware about. I will talk about each of these in detail but this is something that we are going to cover about and you should know about all of these fundamentals. So after that we are going to talk about the firewall and after that there will be gateway and then load balancer. So we are going to talk about all of these things u in this video uh in very detail like how each of [14:33] these work and how the complete flow will look like. Okay. So we will start with basically uh DNS. Now to understand this let's see an example. Okay. So here is my channel on YouTube Mohit Chabra Tech. But if you see this URL out there you will see it says https uh colon/www.youtube.com youtube.com/adtheredmoitabra tech right so let me just copy this and paste it here so that we can discuss about it okay so this is the complete'''

Now glue them together: replace the `{{CHUNK}}` placeholder with the transcript slice.
Then print the size, so you can see how much the model has to read.

In [ ]:
USER_PROMPT = USER_TEMPLATE.replace("{{CHUNK}}", TRANSCRIPT_SLICE)

print("transcript slice:", len(TRANSCRIPT_SLICE), "characters")
print("whole prompt:", len(USER_PROMPT), "characters")
print("roughly", len(USER_PROMPT) // 4, "tokens")    # 1 token is about 4 characters
print()
print("This is what the model will read (first 600 characters):")
print(TRANSCRIPT_SLICE[:600])

## Step 9: Generate

This is the cell that matters. We send the system prompt and the user prompt, and the model writes the answer.

`max_tokens=1500` is the limit for the answer. Our prompt asks for about 180 words, so it should stop well before that.

Look at the **tokens per second** at the bottom. Roughly: 25 or more is good, under 10 means the GPU is not being used.

In [ ]:
answer, tokens, seconds = ask(SYSTEM_PROMPT, USER_PROMPT, max_tokens=1500)

print(answer)
print()
print("-" * 40)
print(f"tokens written: {tokens}")
print(f"time: {seconds:.1f} seconds")
print(f"speed: {tokens / seconds:.1f} tokens per second")

## Step 10: Is the answer valid JSON?

Our pipeline needs real JSON. A broken answer would crash the next stage. So we check.

How the check works:

1. The model sometimes adds a sentence before or after the JSON. So we find the **first `{`** and the **last `}`**
   and cut out what is between them.
2. `json.loads` tries to read that text as JSON. If the text is broken it raises an error, and we catch it.
3. If it worked, we print each topic: its start time, its heading and how many facts it has.

Also read the facts yourself and compare them with the transcript text above.
Are they true? Do they match what was said? Did the model invent anything?

In [ ]:
import json

first_brace = answer.find("{")      # position of the first {  (-1 if there is none)
last_brace = answer.rfind("}")      # position of the last }

try:
    parsed = json.loads(answer[first_brace:last_brace + 1])
    is_valid = True
except Exception as error:
    parsed = None
    is_valid = False
    print("NOT valid JSON:", error)

print("valid JSON:", is_valid)

if is_valid:
    for topic in parsed.get("topics", []):
        print(topic.get("start"), "|", topic.get("heading"), "|", len(topic.get("facts", [])), "facts")

## Step 11: Save this run to Drive

We will compare models and prompts later, so every run is saved: the prompt, the answer, the speed, and whether the JSON was valid.

The file is a `.jsonl`: **one JSON object per line**. Each run adds one line at the end (`"a"` means append),
so nothing is ever overwritten.

In [ ]:
from datetime import datetime

record = {
    "when": datetime.now().isoformat(timespec="seconds"),
    "model": FILE_NAME,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "answer": answer,
    "tokens": tokens,
    "seconds": round(seconds, 1),
    "valid_json": is_valid,
}

with open(RUN_FOLDER + "/digest_tests.jsonl", "a", encoding="utf-8") as file:
    file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved to", RUN_FOLDER + "/digest_tests.jsonl")

## Optional: try your own text

Paste any transcript piece between the triple quotes and run the cell.
It uses the same system prompt and the same template.

In [ ]:
MY_TEXT = """
[00:00] Paste a piece of any lecture transcript here.
"""

my_prompt = USER_TEMPLATE.replace("{{CHUNK}}", MY_TEXT)
my_answer, my_tokens, my_seconds = ask(SYSTEM_PROMPT, my_prompt)

print(my_answer)
print(f"({my_tokens} tokens in {my_seconds:.1f} seconds)")

## What to send back

1. The GPU line from Step 1
2. The two lines from Step 5 (version and "can use the GPU")
3. The speed line from Step 9
4. `valid JSON: True/False` from Step 10, plus the topic lines
5. Your own opinion: were the facts correct?

With that, we decide whether the 8B model is enough or whether to move up to 14B.